<a href="https://colab.research.google.com/github/Terry4715/MVO-backtest/blob/main/toolKit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats
import os
import sys
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive/')

# Define the path to directory
data_path = '/content/drive/MyDrive/ColabNotebooks/EDHEC_Learning'

# Add the directory to sys.path to allow Python to find custom module
if data_path not in sys.path:
    sys.path.append(data_path)

Mounted at /content/drive/


In [ ]:
def get_ls_returns():
  '''
  Load the montly large and small cap dataset for the top and bottle decile returns by market cap
  '''
  # Define the base path for Google Drive
  gdrive_base_path = '/content/drive/MyDrive/'
  # Construct the full path to the data file
  file_path = os.path.join(gdrive_base_path, 'ColabNotebooks', 'EDHEC_Learning', 'data', 'Portfolios_Formed_on_ME_monthly_EW.csv')

  data = pd.read_csv(file_path, header=0, index_col=0, na_values=-99.99)
  rets = data[['Lo 10','Hi 10']]
  rets.index = pd.to_datetime(rets.index, format='%Y%m').to_period('M')
  rets.columns = ['SmallCap','LargeCap']
  rets = rets/100
  return rets


def get_hf_returns():
  '''
  Load the montly hedge fund index returns dataset
  '''
  # Define the base path for Google Drive
  gdrive_base_path = '/content/drive/MyDrive/'
  # Construct the full path to the data file
  file_path = os.path.join(gdrive_base_path, 'ColabNotebooks', 'EDHEC_Learning', 'data', 'edhec-hedgefundindices.csv')

  data = pd.read_csv(file_path, header=0, index_col=0, parse_dates=True, dayfirst=True)
  rets = data/100
  rets.index = rets.index.to_period('M')
  return rets


def get_ind_returns():
  '''
  Load and format the monthy Ken Fresh 30 industry portfolio retutrn dataset
  '''
  # Define the base path for Google Drive
  gdrive_base_path = '/content/drive/MyDrive/'
  # Construct the full path to the data file
  file_path = os.path.join(gdrive_base_path, 'ColabNotebooks', 'EDHEC_Learning', 'data', 'ind30_m_vw_rets.csv')
  dataset = pd.read_csv(file_path, header=0, index_col=0, parse_dates=True, date_format='%Y%m')/100
  dataset.index = pd.to_datetime(dataset.index).to_period('M')
  dataset.columns = dataset.columns.str.strip()
  return dataset


def ann_rets(r, periods_per_year):
  '''
  Annualise a set of returns
  Both return series and periods per year should be provideded
  '''
  compounded_growth = (1+r).prod()
  n_periods = r.shape[0]
  return compounded_growth**(periods_per_year/n_periods)-1


def ann_vol(r, periods_per_year):
  '''
  Annualise the volatility of a set of returns
  Both return series and periods per year should be provideded
  '''
  return r.std()*(periods_per_year**0.5)


def sharpe_ratio(r, riskfree_rate, periods_per_year):
  '''
  Computes the annualised sharpe ratio of a set of returns
  '''
  #convert the annual risk-free rate to per period
  rf_per_period = (1+riskfree_rate)**(1/periods_per_year)-1
  excess_ret = r - rf_per_period
  ann_ex_ret = ann_rets(excess_ret, periods_per_year)
  annualised_vol = ann_vol(r, periods_per_year)
  return ann_ex_ret/annualised_vol


def drawdown(return_series: pd.Series):
  '''
  Takes a time series of asset returns.
  Computes and returns a DataFrame that contains:
  the wealth index
  the previous peaks
  percent drawdowns
  '''
  wealth_index = 1000*(1+return_series).cumprod()
  previous_peaks = wealth_index.cummax()
  drawdowns = (wealth_index - previous_peaks)/previous_peaks
  return pd.DataFrame({"Wealth": wealth_index,
                       "Previous Peak": previous_peaks,
                       "Drawdown": drawdowns})


def semideviation(r):
  '''
  Returns the semideviation aka negative semideviation of r
  r must be a Series or a DataFrame, else raises a TypeError
  '''
  excess= r-r.mean()                                        # We demean the returns
  excess_negative = excess[excess<0]                        # We take only the returns below the mean
  excess_negative_square = excess_negative**2               # We square the demeaned returns below the mean
  n_negative = (excess<0).sum()                             # number of returns under the mean
  return (excess_negative_square.sum()/n_negative)**0.5     # semideviation


def skewness(r):
  '''
  Alt script to scipy.skew()
  Computes the skewness of the supplied Series or DataFrame
  Returns a float or a Series
  '''
  demeaned_r = r - r.mean()
  # use the population std by setting dof=0
  sigma_r = r.std(ddof=0)
  exp = (demeaned_r**3).mean()
  return exp/sigma_r**3


def kurtosis(r):
  '''
  Alt script to scipy.kurtosis()
  Computes the kurtosis of the supplied Series or DataFrame
  Returns a float or a Series
  '''
  demeaned_r = r - r.mean()
  # use the population std by setting dof=0
  sigma_r = r.std(ddof=0)
  exp = (demeaned_r**4).mean()
  return exp/sigma_r**4


def is_normal(r, level=0.01):
  '''
  Applies the Jarque-Bera test to determine if a Series is normal or not
  Test is applied at the 1% level by default
  Returns True if the hypothesis of normality is accepted, False otherwise
  '''
  statistic, p_value = scipy.stats.jarque_bera(r)
  return p_value > level


def var_historic(r, level=5):
  '''
  Value At Risk is the probability at a given level, you will lose X amount across return period
  Returns the historic Value at Risk at a specified level
  i.e. returns the number such that "level" percent of the returns
  fall below that amount of returns, and the (100-level) percent are above
  '''
  if isinstance(r, pd.DataFrame):
    return r.aggregate(var_historic, level=level)
  elif isinstance(r, pd.Series):
    # VaR typically reported as positive number
    return -np.percentile(r, level)
  else:
    raise TypeError("Expected r to be a Series or DataFrame")


def var_gaussian(r, level=5, modified=False):
  '''
  Returns the Parametrix Gaussian VaR of a Series or DataFrame
  If "modified" is True, then the modified VaR is returned,
  using the Cornish-Fisher modification
  '''
  # compute the Z-Score assuming it was Gaussian (normally distributed)
  z = scipy.stats.norm.ppf(level/100)
  if modified:
    # modify the Z-Score based on observed skewness and kurtosis
    s = skewness(r)
    k = kurtosis(r)
    z = (z +
        (z**2 - 1)*s/6 +
        (z**3 - 3*z)*(k-3)/24 -
        (2*z**3 - 5*z)*(s**2)/36)

  return -(r.mean() + z*r.std(ddof=0))


def cvar_historic(r, level=5):
  '''
  Computes the Conditional VaR of Series or DataFrame
  '''
  if isinstance(r, pd.Series):
    is_beyond = r <= -var_historic(r, level=level)
    return -r[is_beyond].mean()
  elif isinstance(r, pd.DataFrame):
    return r.aggregate(cvar_historic, level=level)
  else:
    raise TypeError("Expected r to be a Series or DataFrame")


